# Differential Privacy Training with AdvSecureNet

This notebook demonstrates how to train a model with differential privacy using AdvSecureNet. We'll train a ResNet-18 model on the CIFAR-10 dataset while preserving privacy using the Opacus library.

## What is Differential Privacy?

Differential privacy limits the extent to which the model's output can reveal information about any individual training example.

### Key Parameters:
- **`noise_multiplier`**: Controls the amount of noise added during training. Higher values = more privacy but potentially lower utility
- **`max_grad_norm`**: Maximum L2 norm for gradient clipping. Bounds the sensitivity of the model
- **`delta`**: Privacy parameter that bounds the probability of privacy failure
- **`kwargs`**: Additional Opacus parameters. When using kwargs, ensure that parameter names match exactly what Opacus expects to avoid runtime errors. Compatibility with advsecurenet is not guaranteed for all possible kwargs modifications.

## Step 1: Setup and Imports

In [9]:
# Core imports
import torch
import gc

# AdvSecureNet imports
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.trainer.trainer import Trainer
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.shared.types.configs import TrainConfig
from advnet_common.types.configs.base import (
    CheckpointBase,
    FinalModelBase,
    OptimizationBase,
    DifferentialPrivacyBase
)
from advsecurenet.shared.types.configs.train_config import (
    ModelConfig,
    TrainingProcessConfig,
    DeviceConfig
    
)

/Users/philip/Desktop/advsecurenet_mp/clean_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/philip/Desktop/advsecurenet_mp/advsecurenet/datasets/base_dataset.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/philip/Desktop/advsecurenet_mp/advsecurenet/datasets/base_dataset.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Step 2: Create ResNet-18 Model

Initialize ResNet-18 with the correct number of output classes for CIFAR-10.

In [10]:
# Create ResNet-18 model for CIFAR-10
model = ModelFactory.create_model(
    model_name="resnet18",  
    architecture={"num_classes": 10},  
    pretrained=False 
)

import torch.nn as nn

# Access the underlying model
if hasattr(model, 'model'):
    base_model = model.model
else:
    base_model = model

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
gc.collect()

# Ensure the final layer has exactly 10 classes
if hasattr(base_model, 'fc'):
    in_features = base_model.fc.in_features
    base_model.fc = nn.Linear(in_features, 10)
    print(f"Final layer configured with 10 classes (input features: {in_features})")

# Make the model compatible with Opacus by fixing in-place operations
from opacus.validators import ModuleValidator

def fix_inplace_operations(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.ReLU):
            child.inplace = False
        else:
            fix_inplace_operations(child)

# Apply the manual fix to the underlying model
if hasattr(model, 'model'):
    fix_inplace_operations(model.model)
else:
    fix_inplace_operations(model)

# Apply Opacus validator fix
model = ModuleValidator.fix(model)

print("Loaded ResNet-18 model with 10 classes for CIFAR-10")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

# Verify the final layer configuration
if hasattr(model, 'model') and hasattr(model.model, 'fc'):
    fc_layer = model.model.fc
elif hasattr(model, 'fc'):
    fc_layer = model.fc
else:
    fc_layer = None

if fc_layer:
    print(f"Final layer output classes: {fc_layer.out_features}")
    if fc_layer.out_features == 10:
        print("Model correctly configured for CIFAR-10")
    else:
        print(f"Warning: Model has {fc_layer.out_features} classes instead of 10")
        fc_layer = nn.Linear(fc_layer.in_features, 10)
        if hasattr(model, 'model'):
            model.model.fc = fc_layer
        else:
            model.fc = fc_layer
        print("Fixed: Final layer now has 10 classes")

/Users/philip/Desktop/advsecurenet_mp/advsecurenet/utils/kwargs_utils/filter_kwargs.py:28: UserWarning: Ignoring argument 'num_classes' as it is not accepted by resnet18.
  warnings.warn(


Final layer configured with 10 classes (input features: 512)
Loaded ResNet-18 model with 10 classes for CIFAR-10
Model parameters: 11181642
Final layer output classes: 10
Model correctly configured for CIFAR-10


## Step 3: Setup Data Preprocessing

Configure preprocessing transforms for CIFAR-10 dataset.

In [11]:
# Create preprocessing configuration
preprocess_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(
            name="ToDtype", params={"dtype": "torch.float32", "scale": True}
        ),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        ),
    ]
)

## Step 4: Create CIFAR-10 Dataset and DataLoader

Load the CIFAR-10 dataset and create data loaders for training.

In [12]:
# Create CIFAR-10 dataset
dataset = DatasetFactory.load_dataset(
    dataset_name="cifar10", 
    preprocessing=preprocess_config, 
    num_classes=10
)

train_data = dataset['train']
test_data = dataset['test']

print("CIFAR-10 dataset loaded")
print(f"Training samples: {len(train_data):,}".replace(",", "'"))
print(f"Test samples: {len(test_data):,}".replace(",", "'"))

/Users/philip/Desktop/advsecurenet_mp/clean_venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


CIFAR-10 dataset loaded
Training samples: 50'000
Test samples: 10'000


In [13]:
# Create data loaders
# Note: For differential privacy, batch size should be chosen carefully
train_loader = DataLoaderFactory.create_dataloader(
    dataset=train_data, 
    batch_size=128,
    shuffle=True,
    drop_last=True  # Important for DP: ensures consistent batch sizes
)

print("Data loader created")
print(f"Training batches: {len(train_loader)}")
print(f"Training batch size: {train_loader.batch_size}")

Data loader created
Training batches: 390
Training batch size: 128


## Step 5: Configure Differential Privacy

This is the key step where we configure differential privacy parameters.

### Privacy Parameters Explained:
- **`noise_multiplier=1.2`**: Moderate noise level for reasonable privacy-utility trade-off
- **`max_grad_norm=1.0`**: Standard gradient clipping threshold
- **`delta=1e-5`**: Small probability of privacy failure (< 1 in 100,000)
- **`kwargs`**: Additional Opacus-specific parameters

In [6]:
# Configure differential privacy
# IMPORTANT: When using kwargs, ensure parameter names match Opacus expectations exactly!
differential_privacy_config = DifferentialPrivacyBase(
    enable=True,
    noise_multiplier=0.5,
    max_grad_norm=1.0,
    delta=1e-5,
    kwargs={
        # Example additional parameters (uncomment as needed)
        # "clipping": "flat",     # Gradient clipping method: "flat" or "fast"
        # "loss_reduction": "mean"  # How to reduce loss across samples
    }
)

print("Differential Privacy Configuration:")
print(f"   Enabled: {differential_privacy_config.enable}")
print(f"   Noise Multiplier: {differential_privacy_config.noise_multiplier}")
print(f"   Max Gradient Norm: {differential_privacy_config.max_grad_norm}")
print(f"   Delta: {differential_privacy_config.delta}")
print(f"   Additional kwargs: {differential_privacy_config.kwargs}")
print()
print("NOTE: Higher noise_multiplier = more privacy but potentially lower model performance")

Differential Privacy Configuration:
   Enabled: True
   Noise Multiplier: 0.5
   Max Gradient Norm: 1.0
   Delta: 1e-05
   Additional kwargs: {}

NOTE: Higher noise_multiplier = more privacy but potentially lower model performance


## Step 6: Create Training Configuration

Configure the training process with differential privacy enabled.

In [ ]:
# Training configuration with differential privacy
config = TrainConfig(
    model_config=ModelConfig(model=model),
    training_process_config=TrainingProcessConfig(
        train_loader=train_loader,
        epochs=20,                          
        learning_rate=0.01,               
        criterion="cross_entropy",
        verbose=True,
    ),
    optimization_config=OptimizationBase(
        optimizer="sgd",
        optimizer_kwargs={
            "momentum": 0.9,
            "weight_decay": 1e-4
        }
    ),
    device_config=DeviceConfig(),      
    checkpoint_config=CheckpointBase(),
    final_model_config=FinalModelBase(
        save_final_model=True,
        save_model_path="./models",
        save_model_name="resnet18_cifar10_dp_moderate_privacy"
    ),
    differential_privacy_config=differential_privacy_config,
)

print("Training Configuration:")
print(f"   Epochs: {config.training_process_config.epochs}")
print(f"   Learning Rate: {config.training_process_config.learning_rate}")
print(f"   Optimizer: {config.optimization_config.optimizer}")
print(f"   Differential Privacy: {'ENABLED' if config.differential_privacy_config.enable else 'DISABLED'}")

Training Configuration:
   Epochs: 20
   Learning Rate: 0.01
   Optimizer: sgd
   Differential Privacy: ENABLED


: 

## Step 7: Initialize Trainer and Start Training

Create the trainer and begin differential privacy training.

In [14]:
# Create trainer
trainer = Trainer(config)
print(f"Device: {trainer._device}")
print("Trainer initialized with differential privacy!")
print("Starting training with privacy-preserving guarantees...")

# Start training
trainer.train()

print()
print("Differential privacy training completed successfully!")

NameError: name 'config' is not defined

In [33]:
# Step 8: Evaluate Model Performance on Test Set

def strip_opacus_prefix(state_dict, is_opacus_model=True):
    """
    Strip Opacus '_module.' prefix from state dict keys if needed.
    
    Args:
        state_dict: Model state dictionary
        is_opacus_model: Whether model was trained with Opacus
    
    Returns:
        Cleaned state dictionary
    """
    if not is_opacus_model:
        return state_dict
    
    cleaned_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('_module.'):
            new_key = key[8:]  # Remove '_module.' prefix
            cleaned_state_dict[new_key] = value
        else:
            cleaned_state_dict[key] = value
    
    print(f"Stripped Opacus prefixes from {len(cleaned_state_dict)} parameters")
    return cleaned_state_dict

# Load and evaluate the trained model
import os

model_path = "/Users/philip/Desktop/advsecurenet_mp/examples/advsecurenet/experiments/experiments_files/resnet18_cifar10_pgd_adversarial_trained.pth"

if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location='cpu')  # Load to CPU first
    
    # Create fresh model for evaluation with correct number of classes
    eval_model = ModelFactory.create_model(
        model_name="resnet18", 
        architecture={"num_classes": 10},
        pretrained=False
    )
    
    import torch.nn as nn
    
    if hasattr(eval_model, 'model'):
        base_model = eval_model.model
    else:
        base_model = eval_model
    
    # Ensure the final layer has 10 classes
    if hasattr(base_model, 'fc'):
        in_features = base_model.fc.in_features
        base_model.fc = nn.Linear(in_features, 10)
    
    # Make model Opacus-compatible
    from opacus.validators import ModuleValidator
    
    def fix_inplace_operations(module):
        for name, child in module.named_children():
            if isinstance(child, torch.nn.ReLU):
                child.inplace = False
            else:
                fix_inplace_operations(child)

    if hasattr(eval_model, 'model'):
        fix_inplace_operations(eval_model.model)
    else:
        fix_inplace_operations(eval_model)
    
    eval_model = ModuleValidator.fix(eval_model)
    
    # Strip Opacus prefixes and load weights
    cleaned_state_dict = strip_opacus_prefix(checkpoint, is_opacus_model=True)
    
    try:
        eval_model.load_state_dict(cleaned_state_dict, strict=False)
        print("Model weights loaded successfully")
    except Exception as e:
        print(f"Warning: Model loading failed: {e}")
        print("Using current trainer model instead...")
        eval_model = trainer.model
    
    # Move model to MPS device AFTER loading weights
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    eval_model = eval_model.to(device)
    eval_model.eval()
    
    print(f"Model moved to device: {device}")
    
else:
    print(f"Model file not found: {model_path}")
    print("Using the current trained model for evaluation...")
    eval_model = trainer.model
    device = trainer._device
    eval_model = eval_model.to(device)

Stripped Opacus prefixes from 122 parameters
Model weights loaded successfully
Model moved to device: mps


In [34]:
# Create test data loader and evaluate
test_loader = DataLoaderFactory.create_dataloader(
    dataset=test_data, 
    batch_size=256,
    shuffle=False,
    drop_last=False
)

# Evaluate the model
correct = 0
total = 0
class_correct = [0] * 10
class_total = [0] * 10

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Evaluating trained ResNet-18 model on CIFAR-10 test set...")

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)
        
        outputs = eval_model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Per-class accuracy
        c = (predicted == labels).squeeze()
        for i in range(labels.size(0)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1
        
        if (batch_idx + 1) % 10 == 0:
            print(f"   Processed {batch_idx + 1}/{len(test_loader)} batches...")

# Overall accuracy
overall_accuracy = 100 * correct / total

print(f"\nCIFAR-10 TEST RESULTS:")
print(f"   Test Accuracy: {overall_accuracy:.2f}% ({correct}/{total})")

# Per-class breakdown
print(f"\nPer-Class Accuracy:")
print("   Class Name       | Accuracy")
print("   -----------------|----------")
for i in range(10):
    if class_total[i] > 0:
        class_acc = 100 * class_correct[i] / class_total[i]
        print(f"   {class_names[i]:15} | {class_acc:6.2f}%")

print(f"\nEvaluation completed!")

Evaluating trained ResNet-18 model on CIFAR-10 test set...


/Users/philip/Desktop/advsecurenet_mp/advsecurenet/datasets/base_dataset.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/philip/Desktop/advsecurenet_mp/advsecurenet/datasets/base_dataset.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/philip/Desktop/advsecurenet_mp/advsecurenet/datasets/base_dataset.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setupt

   Processed 10/40 batches...
   Processed 20/40 batches...
   Processed 20/40 batches...
   Processed 30/40 batches...
   Processed 30/40 batches...
   Processed 40/40 batches...
   Processed 40/40 batches...

CIFAR-10 TEST RESULTS:
   Test Accuracy: 25.33% (2533/10000)

Per-Class Accuracy:
   Class Name       | Accuracy
   -----------------|----------
   airplane        |  92.20%
   automobile      |  34.30%
   bird            |  17.90%
   cat             |   3.50%
   deer            |  14.80%
   dog             |   0.20%
   frog            |   0.50%
   horse           |   2.20%
   ship            |  69.90%
   truck           |  17.80%

Evaluation completed!

CIFAR-10 TEST RESULTS:
   Test Accuracy: 25.33% (2533/10000)

Per-Class Accuracy:
   Class Name       | Accuracy
   -----------------|----------
   airplane        |  92.20%
   automobile      |  34.30%
   bird            |  17.90%
   cat             |   3.50%
   deer            |  14.80%
   dog             |   0.20%
   frog    

In [35]:
import torch
import os

def comprehensive_model_validation(model_path):
    """
    Comprehensive validation of model checkpoint structure.
    """
    if not os.path.exists(model_path):
        return {"error": f"File not found: {model_path}"}
    
    try:
        checkpoint = torch.load(model_path, map_location='cpu')
        
        info = {
            "file_exists": True,
            "file_size_mb": os.path.getsize(model_path) / (1024 * 1024),
            "checkpoint_type": type(checkpoint).__name__,
            "all_keys": list(checkpoint.keys()) if isinstance(checkpoint, dict) else [],
            "fc_layers": {},
            "prefix_analysis": {}
        }
        
        # Analyze all keys for prefixes
        all_keys = list(checkpoint.keys()) if isinstance(checkpoint, dict) else []
        
        # Count different prefix patterns
        prefixes = {"model.": 0, "_module.": 0, "_module.model.": 0, "no_prefix": 0}
        
        for key in all_keys:
            if key.startswith('_module.model.'):
                prefixes["_module.model."] += 1
            elif key.startswith('_module.'):
                prefixes["_module."] += 1
            elif key.startswith('model.'):
                prefixes["model."] += 1
            else:
                prefixes["no_prefix"] += 1
        
        info["prefix_analysis"] = prefixes
        
        # Find ALL fc-related keys
        fc_keys = [k for k in all_keys if 'fc' in k]
        
        print(f"\nALL FC-RELATED KEYS in {os.path.basename(model_path)}:")
        for key in fc_keys:
            tensor = checkpoint[key]
            shape = list(tensor.shape) if hasattr(tensor, 'shape') else "No shape"
            print(f"   {key}: {shape}")
            
            info["fc_layers"][key] = {
                "shape": shape,
                "dtype": str(tensor.dtype) if hasattr(tensor, 'dtype') else "Unknown"
            }
        
        # Check for final layer specifically
        final_layer_candidates = [
            'fc.weight', 'model.fc.weight', '_module.fc.weight', 
            '_module.model.fc.weight', 'classifier.weight'
        ]
        
        detected_final_layer = None
        for candidate in final_layer_candidates:
            if candidate in checkpoint:
                detected_final_layer = candidate
                tensor = checkpoint[candidate]
                if hasattr(tensor, 'shape') and len(tensor.shape) == 2:
                    info["final_layer"] = {
                        "key": candidate,
                        "shape": list(tensor.shape),
                        "num_classes": tensor.shape[0],
                        "is_cifar10": tensor.shape[0] == 10,
                        "is_imagenet": tensor.shape[0] == 1000
                    }
                break
        
        return info
        
    except Exception as e:
        return {"error": f"Error loading checkpoint: {str(e)}"}

# Test all models
model_paths = [
    '/Users/philip/Desktop/advsecurenet_mp/examples/advsecurenet/experiments/experiments_files/resnet18_cifar10_pgd_adversarial_trained.pth'
]

print("COMPREHENSIVE MODEL VALIDATION:")
print("=" * 80)

for model_path in model_paths:
    print(f"\nModel: {os.path.basename(model_path)}")
    print("-" * 50)
    
    info = comprehensive_model_validation(model_path)
    
    if "error" in info:
        print(f"   Error: {info['error']}")
        continue
    
    print(f"   File Size: {info['file_size_mb']:.2f} MB")
    print(f"   Total Keys: {len(info['all_keys'])}")
    print(f"   Checkpoint Type: {info['checkpoint_type']}")
    
    print(f"\n   PREFIX ANALYSIS:")
    for prefix, count in info["prefix_analysis"].items():
        if count > 0:
            print(f"      {prefix}: {count} keys")
    
    if "final_layer" in info:
        final = info["final_layer"]
        print(f"\n   FINAL LAYER DETECTED:")
        print(f"      Key: {final['key']}")
        print(f"      Shape: {final['shape']}")
        print(f"      Classes: {final['num_classes']}")
        
        if final['is_cifar10']:
            print(f"      CIFAR-10 Compatible (10 classes)")
        elif final['is_imagenet']:
            print(f"      ImageNet size (1000 classes)")
        else:
            print(f"      Unknown size ({final['num_classes']} classes)")
    else:
        print(f"\n   NO FINAL LAYER DETECTED")

print("\n" + "=" * 80)
print("Validation complete!")

COMPREHENSIVE MODEL VALIDATION:

Model: resnet18_cifar10_pgd_adversarial_trained.pth
--------------------------------------------------

ALL FC-RELATED KEYS in resnet18_cifar10_pgd_adversarial_trained.pth:
   model.fc.weight: [10, 512]
   model.fc.bias: [10]
   File Size: 42.74 MB
   Total Keys: 122
   Checkpoint Type: OrderedDict

   PREFIX ANALYSIS:
      model.: 122 keys

   FINAL LAYER DETECTED:
      Key: model.fc.weight
      Shape: [10, 512]
      Classes: 10
      CIFAR-10 Compatible (10 classes)

Validation complete!
